# Inference over a folder

Runs a checkpoint over every TIFF in a folder and writes two files per image:

    OUT_DIR/masks/<name>.png      raw class ids (a label map, not colors)
    OUT_DIR/overlays/<name>.png   the prediction drawn over the image

No masks or `classes.json` needed — classes are labelled by id.

In [16]:
# ============ 1. SETTINGS ============
IN_DIR    = "../data/combined/val/images"                  # folder of images to run on
OUT_DIR   = "../data/segmentation_workshop_val_results"      # where masks/ and overlays/ are written
#CKPT_PATH = "../checkpoints/historical_plant.ckpt"

CKPT_PATH = "../checkpoints/combined_vits16-eomt-cityscapes-20k.ckpt"

ALPHA = 0.45      # overlay opacity
EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg"}

In [17]:
import os

os.environ["LIGHTLY_TRAIN_CACHE_DIR"] = ".cache"
os.environ["LIGHTLY_TRAIN_MODEL_CACHE_DIR"] = ".cache"
os.environ["TORCH_HOME"] = ".cache"

import colorsys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import lightly_train

Image.MAX_IMAGE_PIXELS = None
IGNORE = 255                  # ignore / unannotated; left unpainted in the overlay
IGNORE_COLOR = (255, 0, 255)


def auto_color(n):
    """A distinct color per class id, for any number of classes."""
    if n < 20:
        return [int(c * 255) for c in plt.get_cmap("tab20")(n)[:3]]
    r, g, b = colorsys.hsv_to_rgb((n * 0.618033988749895) % 1.0, 0.65, 0.95)
    return [int(r * 255), int(g * 255), int(b * 255)]


# Indexed by class id, so a class keeps the same color in every image.
PALETTE = np.array([auto_color(i) for i in range(256)], dtype=np.uint8)
PALETTE[IGNORE] = IGNORE_COLOR

IN_DIR, OUT_DIR = Path(IN_DIR), Path(OUT_DIR)
IMAGES = sorted(p for p in IN_DIR.iterdir()
                if p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith("."))

model = lightly_train.load_model_from_checkpoint(CKPT_PATH)

print(f"input      : {IN_DIR}  ({len(IMAGES)} images)")
print(f"output     : {OUT_DIR}")
print(f"checkpoint : {CKPT_PATH}")

input      : ../data/combined/val/images  (21 images)
output     : ../data/segmentation_workshop_val_results
checkpoint : ../checkpoints/combined_vits16-eomt-cityscapes-20k.ckpt


In [18]:
# ============ 2. RUN INFERENCE AND SAVE ============
def to_uint8_rgb(path, lo=2, hi=98):
    """Read any image as 8-bit RGB.

    Tomography TIFFs are often 16-bit or float, so the 2-98 percentile stretch
    is what makes a faint slice visible.
    """
    a = np.asarray(Image.open(path)).astype(np.float64)
    if a.ndim == 3:
        a = a[..., :3].mean(axis=2)                 # collapse to grayscale
    p, q = np.percentile(a, lo), np.percentile(a, hi)
    a = (np.clip(a, p, q) - p) / (q - p) * 255 if q > p else np.zeros_like(a)
    return np.dstack([a.astype(np.uint8)] * 3)


(OUT_DIR / "masks").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "overlays").mkdir(parents=True, exist_ok=True)

for img_path in IMAGES:
    image = to_uint8_rgb(img_path)

    pred = model.predict(str(img_path))
    pred = pred.detach().cpu().numpy() if hasattr(pred, "detach") else np.asarray(pred)
    pred = pred.astype(np.uint8)

    # blend the class colors over the image, leaving ignore pixels untouched
    color = PALETTE[pred]
    blend = (ALPHA * color + (1 - ALPHA) * image).astype(np.uint8)
    blend[pred == IGNORE] = image[pred == IGNORE]

    # masks hold raw class ids -- reload with np.asarray(Image.open(...))
    Image.fromarray(pred).save(OUT_DIR / "masks" / f"{img_path.stem}.png")
    Image.fromarray(blend).save(OUT_DIR / "overlays" / f"{img_path.stem}.png")

    print(f"  {img_path.name}  ->  classes {sorted(np.unique(pred).tolist())}")

print(f"\ndone -- {len(IMAGES)} images written to {OUT_DIR}")

  earth_Dechert_Dataset1_Try2__141833_76B_1_ming_x00y00_00000_20260804_230312_141833_76B_1_ming_x00y00_00000_0000.png  ->  classes [0, 1, 2, 3, 4, 5, 11, 19, 28, 35, 39, 52, 62, 63]
  earth_MCELHANY_DATASET1_260519_204723_i1s_x00y00_00600_0000.png  ->  classes [1, 5, 6, 7, 8, 9, 10, 28, 34, 35, 36, 39, 63]
  earth_Manga__150503_U1593B1H4W_B_10X_00160_20260804_230315__150503_U1593B1H4W_B_10X_00160_0000.png  ->  classes [1, 7, 8, 10, 11, 12, 13, 28, 29, 34, 35, 36, 37, 39, 52, 63]
  earth_Song_Shale_dataset_1_wse_nanoCT_AI_training_dataset_0000.png  ->  classes [9, 14, 15, 16, 17, 18, 38, 39]
  earth_Song_Shale_dataset_1_wse_nanoCT_AI_training_dataset_0007.png  ->  classes [14, 15, 16, 17, 18, 39]
  earth_Song_Shale_dataset_1_wse_nanoCT_AI_training_dataset_0063.png  ->  classes [14, 15, 16, 17, 18]
  energy_Lang_Ti_PTL_Sample1_Annotated_browse_Sample1_0003.png  ->  classes [19, 20, 21, 22, 52]
  energy_Williams_CO2RGDE_browse_dataset_0003.png  ->  classes [8, 24, 25, 26, 27, 28, 29, 36, 

In [19]:
# ============ 0. COPY THE TEST IMAGES ============
import shutil
from pathlib import Path

SRC = Path("/global/cfs/cdirs/als/segmentation_workshop")
DST = Path("../data/segmentation_workshop_test_images")

STEMS = [
    "anika_20260407_141833_76B_1_ming_x00y00_00025",
    "caetano_recon_20150528_072151_cropped8bit0346",
    "hannah_DROUGHT_10729",
    "hannah_SALT_1_0729",
    "jack_Sample1_8bit_100images",
    "remy_20220615_155934_Sample_41_C-UHTR-MC_01039",
    "worasit_D_1_0230",
]

DST.mkdir(parents=True, exist_ok=True)

for stem in STEMS:
    matches = [p for ext in (".tif", ".tiff") if (p := SRC / f"{stem}{ext}").is_file()]
    if not matches:
        print(f"  MISSING  {stem}")
        continue
    src = matches[0]
    shutil.copy2(src, DST / src.name)
    print(f"  {src.name}")

print(f"\ncopied to {DST}")

  anika_20260407_141833_76B_1_ming_x00y00_00025.tiff
  caetano_recon_20150528_072151_cropped8bit0346.tif
  hannah_DROUGHT_10729.tif
  hannah_SALT_1_0729.tif
  jack_Sample1_8bit_100images.tif
  remy_20220615_155934_Sample_41_C-UHTR-MC_01039.tiff
  worasit_D_1_0230.tif

copied to ../data/segmentation_workshop_test_images
